# 입찰메이트 RFP RAG 시스템

RFP(제안요청서) 문서를 분석하여 핵심 정보를 추출하고 Q&A를 수행하는 RAG 시스템

## 목차
1. 환경 설정 및 패키지 설치
2. 데이터 로드 및 전처리
3. 문서 청킹
4. 임베딩 생성 및 벡터 DB 구축
5. Retrieval 테스트
6. RAG Q&A 시스템
7. 성능 평가
8. 실험 비교

## 1. 환경 설정

In [ ]:
# 패키지 설치 (최초 1회)
# !pip install -r requirements.txt

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# 프로젝트 루트 설정
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# .env 파일에서 API 키 로드 (하드코딩 금지!)
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# API 키 확인
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key or api_key == "your_api_key_here":
    print("!! .env 파일에 OPENAI_API_KEY를 설정해 주세요 !!")
    print(f"   경로: {os.path.join(PROJECT_ROOT, '.env')}")
else:
    print(f"API 키 로드 완료 (끝 4자리: ...{api_key[-4:]})")

from configs.config import Config
from src.document_loader import DocumentLoader
from src.chunker import chunk_documents
from src.embedder import EmbeddingModel, VectorStore
from src.retriever import Retriever
from src.generator import RAGGenerator
from src.evaluator import RAGEvaluator, EVALUATION_QUESTIONS

import pandas as pd

# 설정 초기화 (시나리오 B: OpenAI API 기반)
config = Config(
    scenario="B",
    openai_embedding_model="text-embedding-3-small",
    openai_chat_model="gpt-5-mini",
    chunk_size=800,
    chunk_overlap=200,
    chunking_method="naive",
    retrieval_top_k=5,
    retrieval_method="similarity",
    temperature=0.1,
    vectordb_type="chroma",
)

print(f"\n시나리오: {config.scenario}")
print(f"임베딩 모델: {config.embedding_model}")
print(f"생성 모델: {config.chat_model}")
print(f"청킹: {config.chunking_method} (size={config.chunk_size}, overlap={config.chunk_overlap})")
print(f"검색: {config.retrieval_method} (top_k={config.retrieval_top_k})")

## 2. 데이터 로드 및 전처리

`data/documents/` 디렉토리에 RFP 문서(PDF, HWP)를 넣고, `data/data_list.csv`에 메타데이터를 준비하세요.

In [ ]:
# 메타데이터 확인
metadata_path = os.path.join(PROJECT_ROOT, config.metadata_csv)
if os.path.exists(metadata_path):
    metadata_df = pd.read_csv(metadata_path)
    print(f"메타데이터: {len(metadata_df)}건")
    print(f"컬럼: {list(metadata_df.columns)}")
    display(metadata_df.head())
else:
    metadata_df = None
    print(f"메타데이터 파일이 없습니다: {metadata_path}")
    print("data/data_list.csv 파일을 준비해 주세요.")

In [ ]:
# 문서 로드 (PDF, HWP 모두 지원)
loader = DocumentLoader(
    documents_dir=os.path.join(PROJECT_ROOT, config.documents_dir),
    metadata_csv=metadata_path if metadata_df is not None else None,
)

print("문서 로드 중...")
documents = loader.load_all()

# 로드된 문서 통계
if documents:
    total_chars = sum(len(d["text"]) for d in documents)
    avg_chars = total_chars / len(documents)
    print(f"\n총 문자 수: {total_chars:,}")
    print(f"평균 문자 수: {avg_chars:,.0f}")
    
    # 샘플 확인
    print(f"\n첫 번째 문서 미리보기 (처음 500자):")
    print(documents[0]["text"][:500])
    print(f"\n메타데이터: {documents[0]['metadata']}")
else:
    print("로드된 문서가 없습니다. data/documents/ 디렉토리에 PDF/HWP 파일을 넣어주세요.")

## 3. 문서 청킹

두 가지 청킹 방식 비교:
- **Naive**: 고정 크기 기반 (RecursiveCharacterTextSplitter)
- **Semantic**: RFP 문서 섹션 구조를 인식하여 의미 단위 분할

In [ ]:
# Naive 청킹
print("=" * 50)
print("Naive 청킹 (chunk_size=800, overlap=200)")
print("=" * 50)
naive_chunks = chunk_documents(
    documents,
    method="naive",
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
)

chunk_lengths = [len(c["text"]) for c in naive_chunks]
print(f"\n청크 길이 통계:")
print(f"  평균: {sum(chunk_lengths)/len(chunk_lengths):.0f} chars")
print(f"  최소: {min(chunk_lengths)} chars")
print(f"  최대: {max(chunk_lengths)} chars")

In [ ]:
# Semantic 청킹 (비교용)
print("=" * 50)
print("Semantic 청킹 (RFP 구조 인식)")
print("=" * 50)
semantic_chunks = chunk_documents(
    documents,
    method="semantic",
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
)

sem_lengths = [len(c["text"]) for c in semantic_chunks]
print(f"\n청크 길이 통계:")
print(f"  평균: {sum(sem_lengths)/len(sem_lengths):.0f} chars")
print(f"  최소: {min(sem_lengths)} chars")
print(f"  최대: {max(sem_lengths)} chars")

print(f"\n비교: Naive {len(naive_chunks)}개 vs Semantic {len(semantic_chunks)}개")

In [ ]:
# 사용할 청킹 방식 선택
USE_CHUNKS = naive_chunks  # 또는 semantic_chunks
print(f"선택된 청크 수: {len(USE_CHUNKS)}개")

## 4. 임베딩 생성 및 벡터 DB 구축

In [ ]:
# 임베딩 모델 및 벡터 스토어 초기화
embedding_model = EmbeddingModel(config)
vector_store = VectorStore(config, embedding_model)
vector_store.initialize(collection_name="rfp_documents")

existing_count = vector_store.get_collection_count()
if existing_count > 0:
    print(f"기존 벡터 DB에 {existing_count}개 문서가 있습니다.")
    print("새로 인덱싱하려면 data/vectordb 디렉토리를 삭제 후 다시 실행하세요.")
else:
    print("벡터 DB에 문서를 추가합니다...")
    vector_store.add_documents(USE_CHUNKS)
    print(f"\n최종 저장 문서 수: {vector_store.get_collection_count()}")

## 5. Retrieval 테스트

다양한 검색 방법 비교: Similarity / MMR / Hybrid

In [ ]:
# Retriever 초기화
retriever = Retriever(config, vector_store, embedding_model)

test_query = "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘."
print(f"쿼리: {test_query}\n")
print("=" * 60)

# 1) 유사도 검색
print("\n[1] Similarity Search")
sim_results = retriever.similarity_search(test_query, top_k=3)
for i, r in enumerate(sim_results, 1):
    meta = r.get("metadata", {})
    print(f"  #{i} (score: {r.get('score', 'N/A'):.4f})")
    print(f"      출처: {meta.get('filename', 'N/A')}")
    print(f"      내용: {r['text'][:150]}...")

# 2) MMR 검색
print("\n[2] MMR Search (lambda=0.5)")
mmr_results = retriever.mmr_search(test_query, top_k=3)
for i, r in enumerate(mmr_results, 1):
    meta = r.get("metadata", {})
    print(f"  #{i}")
    print(f"      출처: {meta.get('filename', 'N/A')}")
    print(f"      내용: {r['text'][:150]}...")

# 3) Hybrid 검색
print("\n[3] Hybrid Search (keyword_weight=0.3)")
hybrid_results = retriever.hybrid_search(test_query, top_k=3)
for i, r in enumerate(hybrid_results, 1):
    meta = r.get("metadata", {})
    print(f"  #{i} (hybrid_score: {r.get('hybrid_score', 'N/A'):.4f})")
    print(f"      출처: {meta.get('filename', 'N/A')}")
    print(f"      내용: {r['text'][:150]}...")

## 6. RAG Q&A 시스템

대화 맥락을 유지하며 RFP 문서 기반 Q&A를 수행합니다.

In [ ]:
# RAG Generator 초기화
rag = RAGGenerator(config, retriever)

# ── 대화 1: 단일 문서 질문 + 후속 질문 ──
print("=" * 60)
print("대화 1: 단일 문서 질문 + 후속 질문")
print("=" * 60)

q1 = "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘."
print(f"\n[사용자] {q1}\n")
result1 = rag.generate(q1)
print(f"[AI] {result1['answer']}")
print(f"\n응답시간: {result1['elapsed_time']:.2f}s | 검색문서: {len(result1['retrieved_docs'])}개")

q2 = "콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘."
print(f"\n{'─'*60}")
print(f"\n[사용자] {q2}\n")
result2 = rag.generate(q2)
print(f"[AI] {result2['answer']}")
print(f"\n응답시간: {result2['elapsed_time']:.2f}s | 검색문서: {len(result2['retrieved_docs'])}개")

In [ ]:
# ── 대화 2: 다중 문서 비교 ──
rag.reset_memory()
print("=" * 60)
print("대화 2: 다중 문서 비교")
print("=" * 60)

q3 = "고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래?"
print(f"\n[사용자] {q3}\n")
result3 = rag.generate(q3)
print(f"[AI] {result3['answer']}")
print(f"\n응답시간: {result3['elapsed_time']:.2f}s")

q4 = "고려대학교랑 광주과학기술원 각각 응답 시간에 대한 요구사항이 있나? 문서를 기반으로 정확하게 답변해 줘."
print(f"\n{'─'*60}")
print(f"\n[사용자] {q4}\n")
result4 = rag.generate(q4)
print(f"[AI] {result4['answer']}")
print(f"\n응답시간: {result4['elapsed_time']:.2f}s")

In [ ]:
# ── 대화 3: 문서 외 질문 (모른다고 답해야 함) ──
rag.reset_memory()
print("=" * 60)
print("대화 3: 문서 외 질문 (모른다고 답해야 함)")
print("=" * 60)

q5 = "삼성전자가 발주한 반도체 설계 자동화 사업의 요구사항을 알려줘."
print(f"\n[사용자] {q5}\n")
result5 = rag.generate(q5)
print(f"[AI] {result5['answer']}")
print(f"\n응답시간: {result5['elapsed_time']:.2f}s")

## 7. 성능 평가

LLM-as-a-Judge 방식으로 답변 품질 자동 평가

**평가 지표:** Relevance, Accuracy, Faithfulness, Completeness, Conciseness (각 1-5점), Keyword Recall, Response Time

In [ ]:
# 평가 실행
evaluator = RAGEvaluator(config, generator=rag)

print("전체 평가 세트 실행 중...")
print("(각 질문에 대해 RAG 답변 생성 + LLM Judge 평가)\n")
eval_df = evaluator.run_evaluation_suite(use_llm_judge=True)

display(eval_df)

In [ ]:
# 평가 결과 요약
report = evaluator.summary_report(eval_df)

print("=" * 60)
print("평가 결과 요약")
print("=" * 60)
print(f"\n총 질문 수: {report['total_questions']}")
print(f"평균 응답 시간: {report['avg_elapsed_time']:.2f}s")
print(f"최대 응답 시간: {report['max_elapsed_time']:.2f}s")

if "avg_keyword_recall" in report:
    print(f"평균 키워드 재현율: {report['avg_keyword_recall']:.1%}")

llm_keys = [k for k in report if k.startswith("avg_llm_")]
if llm_keys:
    print(f"\nLLM Judge 점수 (1-5점):")
    for k in llm_keys:
        name = k.replace("avg_llm_", "").title()
        print(f"  {name}: {report[k]:.2f}")

print(f"\n카테고리별 성능:")
for cat, cat_report in report.get("by_category", {}).items():
    print(f"  [{cat}] 질문 {cat_report['count']}개, 평균 {cat_report['avg_elapsed_time']:.2f}s")

evaluator.save_results()

## 8. 실험 비교

다양한 Retrieval 설정 조합을 비교합니다.

In [ ]:
import time

experiments = [
    {"name": "baseline_sim_k3", "method": "similarity", "top_k": 3},
    {"name": "baseline_sim_k5", "method": "similarity", "top_k": 5},
    {"name": "baseline_sim_k10", "method": "similarity", "top_k": 10},
    {"name": "mmr_k5", "method": "mmr", "top_k": 5},
    {"name": "hybrid_k5", "method": "hybrid", "top_k": 5},
]

test_queries = [
    "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘.",
    "한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데, 이 사업이 왜 추진되는지 목적을 알려 줘.",
    "교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?",
]

experiment_results = []

for exp in experiments:
    print(f"\n{'='*50}")
    print(f"실험: {exp['name']}")
    
    exp_times = []
    for query in test_queries:
        rag.reset_memory()
        start = time.time()
        result = rag.generate(query, retrieval_method=exp["method"], top_k=exp["top_k"])
        elapsed = time.time() - start
        exp_times.append(elapsed)
        print(f"  {query[:40]}... -> {elapsed:.2f}s")
    
    experiment_results.append({
        "experiment": exp["name"],
        "method": exp["method"],
        "top_k": exp["top_k"],
        "avg_time": sum(exp_times) / len(exp_times),
        "max_time": max(exp_times),
    })

exp_df = pd.DataFrame(experiment_results)
print(f"\n{'='*50}")
print("실험 결과 비교")
display(exp_df)

## 9. 인터랙티브 Q&A (자유 질문)

In [ ]:
# 대화 초기화 후 자유 질문
rag.reset_memory()

question = "기초과학연구원 극저온시스템 사업 요구에서 AI 기반 예측에 대한 요구사항이 있나?"

result = rag.generate(question, stream=True)
print(f"\n\n응답시간: {result['elapsed_time']:.2f}s")
print(f"참조 문서:")
for i, doc in enumerate(result["retrieved_docs"], 1):
    meta = doc.get("metadata", {})
    print(f"  {i}. {meta.get('filename', 'N/A')} (score: {doc.get('score', 'N/A'):.4f})")

In [ ]:
# 후속 질문 (대화 맥락 유지 - reset_memory() 호출하지 않음)
follow_up = "그럼 모니터링 업무에 대한 요청사항이 있는지 찾아보고 알려 줘."

result = rag.generate(follow_up, stream=True)
print(f"\n\n응답시간: {result['elapsed_time']:.2f}s")